# Week 10 — Tuesday Lab Notebook  
## Topic: Building a Model with PyTorch

This lab is for **Week 10 Tuesday**.  
Today we will move from theory to **hands-on deep learning model building** in very simple wording.

We will cover:

- what a tensor is
- why tensors are used in deep learning
- dataset and dataloader idea
- converting data into tensors
- defining a neural network model
- the **forward pass**
- loss function
- optimizer
- training loop
- evaluation on test data
- saving and loading a model checkpoint

Today our goal is simple:

**Students should see the complete journey from raw data to a trained neural network model.**


## 3-Hour Structure

**Hour 1**
- revise what a neural network needs as input
- understand tensors
- understand dataset and dataloader
- prepare a small dataset for training

**Hour 2**
- define a PyTorch model
- understand `forward()`
- understand loss and optimizer
- run a single training step manually

**Hour 3**
- build the full training loop
- evaluate the model
- save and load checkpoint
- discuss common beginner mistakes


## Part A — Why We Need a Framework Today

Yesterday we discussed neurons, weights, bias, activations, and learning.

But in real projects, we need a tool that can help us:

- handle tensors
- define layers
- calculate gradients automatically
- update weights
- train model in batches

For Tuesday lab, we will use **PyTorch** because:

- it is readable
- it feels close to normal Python
- students can understand the steps more clearly
- it is very common in deep learning education and research


## Part B — What is a Tensor?

A **tensor** is the main data structure in deep learning.

Very simple understanding:

- a single number is like a **scalar**
- a list of numbers is like a **vector**
- a table of numbers is like a **matrix**
- tensors are a more general form that can represent all of these

So in practice:

- 0D tensor = one value
- 1D tensor = one row of values
- 2D tensor = table
- 3D or higher = image batches, sequence batches, and more

In PyTorch, tensors are like NumPy arrays, but they are designed for deep learning work.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)


## Part C — Creating Simple Tensors

Let us create a few small tensors and inspect them.


In [ ]:
scalar = torch.tensor(7)
vector = torch.tensor([10, 20, 30, 40])
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])

print("Scalar:", scalar)
print("Vector:\n", vector)
print("Matrix:\n", matrix)


In [ ]:
print("Scalar shape:", scalar.shape)
print("Vector shape:", vector.shape)
print("Matrix shape:", matrix.shape)

print("Vector dtype:", vector.dtype)
print("Matrix ndim:", matrix.ndim)


### Teaching point

Students should observe:

- tensor values look similar to arrays
- `shape` tells the size
- `ndim` tells how many dimensions there are
- deep learning models expect data in tensor form


## Part D — Tensor from NumPy Array

In many real projects, data first comes from:

- CSV
- Excel
- Pandas
- NumPy

Then we convert it into PyTorch tensors.


In [ ]:
arr = np.array([[1.5, 2.0], [3.2, 4.8], [5.1, 6.4]], dtype=np.float32)
tensor_from_numpy = torch.tensor(arr)

print("NumPy array:\n", arr)
print("\nTensor:\n", tensor_from_numpy)
print("\nTensor dtype:", tensor_from_numpy.dtype)


## Part E — Reshaping Tensors

Sometimes the model expects data in a certain shape.

So students should know basic shape operations such as:

- `reshape()`
- `view()` in some contexts
- `unsqueeze()`
- `squeeze()`


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0])

print("Original:", x)
print("Original shape:", x.shape)

x_col = x.reshape(4, 1)
print("\nReshaped to column:\n", x_col)
print("New shape:", x_col.shape)

x_extra = x.unsqueeze(0)
print("\nAfter unsqueeze(0):", x_extra)
print("Shape:", x_extra.shape)


### Simple explanation

- `reshape(4, 1)` changes a 1D tensor into 4 rows and 1 column
- `unsqueeze(0)` adds one extra dimension
- shape matters because neural networks expect data in specific form


## Part F — Dataset Idea

A **dataset** means the full collection of input data and target values.

For example:

- inputs = student study data
- target = pass or fail

Or:

- inputs = patient information
- target = disease yes/no

In today's lab we will use a ready-made dataset from scikit-learn so students can focus on the **model building process**.


## Part G — Load a Small Classification Dataset

We will use the **Breast Cancer** dataset from scikit-learn.

Why use it here?

- it is already available
- it is numeric
- it is good for binary classification
- it works well for beginner neural network labs


In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target classes:", np.unique(y))
print("First 5 feature names:", data.feature_names[:5])


In [ ]:
print("First sample features:\n", X[0][:10])
print("\nFirst sample target:", y[0])


### How to explain the target

This is a **binary classification** problem.

That means the output has two classes only.

So our neural network will learn to predict one of two classes.


## Part H — Train/Test Split

We should not train and test on the same data.

So first we split the dataset into:

- training data
- testing data

This helps us judge whether the model learned properly.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## Part I — Why Scaling is Important

Neural networks usually train better when numeric features are on a similar scale.

If one feature is very large and another is very small, training can become unstable or slow.

So we will use `StandardScaler`.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training data shape:", X_train_scaled.shape)
print("Scaled test data shape:", X_test_scaled.shape)


## Part J — Convert Data into PyTorch Tensors

Now we convert NumPy arrays into tensors.

Important beginner point:

- input features should usually be floating point
- binary targets often need float when using `BCEWithLogitsLoss`


In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)


### Teaching point

Notice the target shape:

- before reshaping, target is like `(455,)`
- after reshaping, it becomes `(455, 1)`

This is useful because our model output will also be one column for binary classification.


## Part K — What is `TensorDataset`?

A `TensorDataset` simply keeps input tensors and target tensors together.

It pairs:

- one row of features
- one target value

This is helpful when training in batches.


In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

print("Training dataset length:", len(train_dataset))
print("Testing dataset length:", len(test_dataset))
print("One sample from train_dataset:")
print(train_dataset[0])


## Part L — What is a `DataLoader`?

A `DataLoader` helps us:

- read data in batches
- shuffle training data
- make training easier

Instead of giving the full dataset at once, we often give smaller groups called **batches**.


In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Number of training batches:", len(train_loader))
print("Number of testing batches:", len(test_loader))


In [ ]:
features_batch, labels_batch = next(iter(train_loader))

print("Batch feature shape:", features_batch.shape)
print("Batch label shape:", labels_batch.shape)


### Simple explanation

If batch size is `32`, each step of training sees 32 samples at a time.

This is more practical than:

- one sample at a time
- or the full dataset at once


## Part M — Define the Neural Network Model

Now we will build a very small neural network.

Our model will have:

- input layer = number of features
- hidden layer 1
- hidden layer 2
- output layer = 1 value for binary classification


In [ ]:
input_size = X_train_tensor.shape[1]
input_size


In [ ]:
class SimpleBinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

model = SimpleBinaryClassifier(input_size)
print(model)


### Very important teaching point

We did **not** put `Sigmoid()` in the final layer here.

Why?

Because we plan to use **`BCEWithLogitsLoss`**, and it already handles the sigmoid behavior internally in a stable way.

This is a very useful beginner lesson.


## Part N — What is the `forward()` Function?

The `forward()` function tells PyTorch:

**when input comes in, how should it pass through the network?**

So:

- input enters layer 1
- activation is applied
- then layer 2
- activation again
- then final output

That full flow is the **forward pass**.


In [ ]:
sample_output = model(features_batch)
print("Model raw output shape:", sample_output.shape)
print("First 5 raw outputs:\n", sample_output[:5])


### How to explain this output

These are **raw scores**, also called **logits**.

They are not yet final class labels.

Later we can convert them into probabilities using sigmoid.


## Part O — Convert Logits to Probabilities

For binary classification:

- raw output = logits
- apply sigmoid = probability between 0 and 1


In [ ]:
probabilities = torch.sigmoid(sample_output)

print("First 5 probabilities:\n", probabilities[:5])


In [ ]:
predicted_classes = (probabilities >= 0.5).float()
print("First 5 predicted classes:\n", predicted_classes[:5])
print("First 5 true labels:\n", labels_batch[:5])


## Part P — Loss Function

A **loss function** tells the model how wrong it is.

If prediction is poor, loss is high.  
If prediction is better, loss becomes lower.

For binary classification with logits, a common choice is:

**`BCEWithLogitsLoss()`**


In [ ]:
criterion = nn.BCEWithLogitsLoss()

loss = criterion(sample_output, labels_batch)
print("Loss on one batch:", loss.item())


## Part Q — Optimizer

The optimizer updates the model weights so the loss can reduce over time.

Today we will use **Adam** because it is common and beginner-friendly.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer


## Part R — One Manual Training Step

Before writing the full loop, students should see one step manually.

The common order is:

1. zero old gradients
2. forward pass
3. compute loss
4. backward pass
5. optimizer step


In [ ]:
features_batch, labels_batch = next(iter(train_loader))

optimizer.zero_grad()

outputs = model(features_batch)
loss = criterion(outputs, labels_batch)

loss.backward()
optimizer.step()

print("Loss after one training step:", loss.item())


### What each line means

- `optimizer.zero_grad()` removes old stored gradients
- `model(features_batch)` performs forward pass
- `criterion(outputs, labels_batch)` calculates loss
- `loss.backward()` calculates gradients
- `optimizer.step()` updates weights


## Part S — Full Training Loop

Now we repeat that process for many epochs.

An **epoch** means one full pass over the training dataset.


In [ ]:
num_epochs = 30
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_features, batch_labels in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")


## Part T — Plot the Training Loss

A simple loss curve helps students see whether training is moving in the right direction.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o")
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()


### Teaching point

If the loss generally goes down, the model is learning.

But students should also remember:

- low training loss alone is not enough
- we still need evaluation on unseen data


## Part U — Model Evaluation on Test Data

During evaluation, we do not update weights.

So we use:

- `model.eval()`
- `torch.no_grad()`


In [ ]:
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        logits = model(batch_features)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        all_preds.extend(preds.numpy().flatten())
        all_true.extend(batch_labels.numpy().flatten())

accuracy = accuracy_score(all_true, all_preds)
print("Test accuracy:", round(accuracy, 4))


In [ ]:
cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:\n", cm)


In [ ]:
print(classification_report(all_true, all_preds))


## Part V — Why `train()` and `eval()` Both Matter

PyTorch models can behave differently in training and evaluation mode, especially when layers like:

- dropout
- batch normalization

are used.

So it is a good habit to write:

- `model.train()` during training
- `model.eval()` during evaluation


## Part W — Predicting Probabilities for a Few Test Samples

Let us inspect a few real predictions.


In [ ]:
model.eval()

with torch.no_grad():
    small_batch = X_test_tensor[:5]
    small_logits = model(small_batch)
    small_probs = torch.sigmoid(small_logits)
    small_preds = (small_probs >= 0.5).float()

print("Probabilities:\n", small_probs.numpy().round(4))
print("\nPredicted classes:\n", small_preds.numpy().astype(int))
print("\nTrue classes:\n", y_test_tensor[:5].numpy().astype(int))


## Part X — Save the Model Checkpoint

After training, we often save the learned parameters.

This is useful because:

- we do not need to retrain every time
- we can use the model later
- we can share the saved model


In [ ]:
checkpoint_path = "week10_tuesday_simple_binary_classifier.pth"
torch.save(model.state_dict(), checkpoint_path)

print("Model saved to:", checkpoint_path)


## Part Y — Load the Saved Model

To load the model again:

- create the same model structure
- load saved weights


In [ ]:
loaded_model = SimpleBinaryClassifier(input_size)
loaded_model.load_state_dict(torch.load(checkpoint_path))
loaded_model.eval()

print("Checkpoint loaded successfully.")


In [ ]:
with torch.no_grad():
    loaded_logits = loaded_model(X_test_tensor[:5])
    loaded_probs = torch.sigmoid(loaded_logits)
    loaded_preds = (loaded_probs >= 0.5).float()

print("Loaded model predictions:\n", loaded_preds.numpy().astype(int))


## Part Z — A Tiny Look at Batches

Students sometimes ask:

**Why not train on the whole dataset at once?**

Because batches help with:

- memory efficiency
- faster updates
- practical training workflow

Let us compare number of batches for different batch sizes.


In [ ]:
for bs in [8, 16, 32, 64, 128]:
    loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    print(f"Batch size {bs}: {len(loader)} batches")


## Part AA — Common Shape Mistakes

Deep learning beginners often face errors because of:

- wrong input shape
- wrong target shape
- wrong data type

So always check:

- `x.shape`
- `y.shape`
- `x.dtype`
- `y.dtype`


In [ ]:
print("Input dtype:", X_train_tensor.dtype)
print("Target dtype:", y_train_tensor.dtype)
print("Input shape:", X_train_tensor.shape)
print("Target shape:", y_train_tensor.shape)


### One common mistake

If the target shape is `(455,)` but output shape is `(455, 1)`, loss calculation may become confusing.

That is why we used:

`y_train_tensor.reshape(-1, 1)`


## Part AB — Common Beginner Errors in PyTorch

1. forgetting to scale numeric features  
2. forgetting `optimizer.zero_grad()`  
3. using wrong target shape  
4. using wrong loss function for the task  
5. mixing probabilities and logits  
6. forgetting `model.eval()` during testing  
7. thinking one epoch is enough for every problem  
8. comparing models without fixing random seed or setup


## Part AC — Optional Exposure: Very Small Keras Version

Today our main practice is PyTorch.  
But students may ask how the same thing looks in Keras.

This cell is only for exposure.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense

    keras_model = Sequential([
        Dense(16, activation="relu", input_shape=(input_size,)),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    keras_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    keras_model.summary()
except Exception as e:
    print("Keras example not run:", e)


### Teaching point

Keras is shorter and very convenient.

But PyTorch shows the training process more openly, which is why it is excellent for teaching Tuesday's topic: **building a model step by step**.


## Part AD — Mini In-Class Practice

Do these in class:

1. Change hidden layer sizes from `(16, 8)` to `(32, 16)` and retrain  
2. Change learning rate from `0.001` to `0.01` and observe loss  
3. Change batch size from `32` to `64`  
4. Train for only 10 epochs and compare result  
5. Print predictions for 10 test samples and compare with true labels


## Part AE — 10 After-Lab Tasks

### Task 1
Create a scalar, a vector, and a 2D tensor in PyTorch. Print their shapes.

### Task 2
Convert a NumPy array into a PyTorch tensor and print its dtype.

### Task 3
Load any small dataset from scikit-learn and print the feature matrix shape and target shape.

### Task 4
Split a dataset into train and test parts using `train_test_split()`.

### Task 5
Scale the training and testing input data using `StandardScaler`.

### Task 6
Convert scaled input data and target data into PyTorch tensors.

### Task 7
Create a `TensorDataset` and a `DataLoader` with batch size `16`.

### Task 8
Define a small neural network with one hidden layer using `nn.Module`.

### Task 9
Write one manual training step using:
- `zero_grad()`
- forward pass
- loss
- backward pass
- optimizer step

### Task 10
Train the model for multiple epochs, print loss after each epoch, and calculate test accuracy.


## Part AF — Optional Homework Challenge

Use a classification dataset of your choice from scikit-learn.

Do the following:

1. load the dataset  
2. inspect features and target  
3. split data into train and test  
4. scale input features  
5. convert data into tensors  
6. create dataset and dataloader  
7. build a neural network in PyTorch  
8. train it for at least 20 epochs  
9. evaluate accuracy on test set  
10. save the model checkpoint  
11. load the checkpoint again  
12. write a short paragraph explaining what you learned


## Part AG — Final Revision Notes

Today you learned:

- tensors are the main data structure in PyTorch
- data often moves from NumPy/Pandas into tensors
- `TensorDataset` keeps features and labels together
- `DataLoader` gives batches during training
- a PyTorch model is often defined using `nn.Module`
- `forward()` explains how input flows through the network
- the loss function tells how wrong predictions are
- the optimizer updates model weights
- the training loop repeats learning over epochs
- `model.eval()` is used during evaluation
- checkpoints help us save and reuse trained models
